# 第 14 节：Actor-Critic 方法

## 📍 位置
Baseline (13) → **Actor-Critic (14)** → A2C (15) → ...

## 🎯 学习目标
1. 从 REINFORCE-with-Baseline 过渡到 Actor-Critic
2. 理解 TD error 作为 advantage 的估计
3. 掌握 detach 的必要性
4. 实现 one-step Actor-Critic
5. 理解 semi-gradient 的含义

## 1. 从 REINFORCE+Baseline 到 Actor-Critic

### REINFORCE+Baseline 的局限

即使加了 baseline，REINFORCE 仍然需要**等 episode 结束**才能计算 $G_t$。
这意味着：
- 不能在线学习（必须按 episode 更新）
- 不能用于 continuing tasks
- 方差仍然较高

### Actor-Critic 的改进

**用 bootstrapping 替代 Monte Carlo return！**

$$\text{REINFORCE: } G_t = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots$$
$$\text{Actor-Critic: } \hat{G}_t = r_{t+1} + \gamma V(s_{t+1})$$

这就是 **TD(0) target**！

## 2. Actor 与 Critic 的职责

| 组件 | 网络 | 输入 | 输出 | 用途 |
|------|------|------|------|------|
| **Actor** | π_θ(a\|s) | 状态 s | 动作分布 | 决定做什么 |
| **Critic** | V_w(s) | 状态 s | 标量价值 | 评价 Actor 做得好不好 |

### Advantage 函数

$$A(s, a) = Q(s, a) - V(s)$$

Advantage 衡量动作 $a$ 比平均好多少。

### TD Error = Unbiased Estimate of Advantage

$$\mathbb{E}[\delta_t | s_t, a_t] = \mathbb{E}[r_{t+1} + \gamma V(s_{t+1}) - V(s_t) | s_t, a_t] = Q(s_t, a_t) - V(s_t) = A(s_t, a_t)$$

**TD error 是 advantage 的无偏估计！**（虽然 $V$ 有估计误差时会有偏）

## 3. One-Step Actor-Critic 算法

### 伪代码
```
初始化 Actor π_θ 和 Critic V_w
对每个 episode:
    初始化状态 s
    重复:
        从 π_θ(·|s) 采样动作 a
        执行 a，观察到 r, s'
        δ = r + γ V(s') - V(s)         ← TD error (advantage 估计)
        
        # Critic 更新 (semi-gradient)
        w ← w + α_w · δ · ∇_w V(s)
        
        # Actor 更新
        θ ← θ + α_θ · δ · ∇_θ log π_θ(a|s)
        #         ↑ 注意：用 δ.detach()!
        
        s ← s'
    直到 episode 终止
```

### 关键细节：detach TD error

```python
td_error = reward + gamma * V(next_state).detach() - V(state)
#                                 ^^^^^^^^
# 不对 target 中的 V 求梯度！
```

这就是 **semi-gradient**：只对当前估计求梯度，不对 target 求梯度。

In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np
import torch; import torch.nn as nn; import torch.optim as optim
import gymnasium as gym
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import os

FIG_DIR = 'outputs/figures'; os.makedirs(FIG_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 4. 从零实现 One-Step Actor-Critic

In [ ]:
class ActorCritic:
    """One-Step Actor-Critic 从零实现

    核心更新:
        δ = r + γ V(s') - V(s)          (TD error)
        w ← w + α_w · δ · ∇V(s)          (Critic)
        θ ← θ + α_θ · δ · ∇log π(a|s)    (Actor)
    """

    def __init__(self, state_dim, n_actions, hidden_dim=128, actor_lr=1e-3, critic_lr=1e-3, gamma=0.99):
        self.gamma = gamma
        self.n_actions = n_actions

        # Actor 网络: 输出动作 logits
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        ).to(DEVICE)

        # Critic 网络: 输出 V(s) 标量
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        ).to(DEVICE)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)

        # 指标跟踪
        self.actor_losses = []
        self.critic_losses = []
        self.entropies = []

    def act(self, state, train=True):
        """采样动作并返回 log_prob"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)  # (1, D)
        logits = self.actor(state_t)  # (1, A)
        probs = torch.softmax(logits, dim=-1)

        if not train:
            return torch.argmax(probs, dim=-1).item()

        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        return action.item(), log_prob

    def get_value(self, state):
        """Critic 估计 V(s)"""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        return self.critic(state_t).squeeze(-1)  # (1,)

    def update(self, state, action_log_prob, reward, next_state, terminated):
        """单步 Actor-Critic 更新

        Args:
            state: 当前状态 (D,)
            action_log_prob: log π(a|s) — tensor
            reward: 即时奖励
            next_state: 下一状态 (D,)
            terminated: 是否终止 (terminated, not truncated)
        """
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        next_state_t = torch.FloatTensor(next_state).unsqueeze(0).to(DEVICE)

        # 当前 V(s)
        V_s = self.critic(state_t).squeeze(-1)  # (1,)

        # TD target: r + γ V(s') (如果 done 则不 bootstrap)
        with torch.no_grad():
            V_s_next = self.critic(next_state_t).squeeze(-1)  # (1,)
            td_target = reward + self.gamma * V_s_next * (1 - float(terminated))

        # TD error (detach — 不对 target 求梯度!)
        td_error = td_target - V_s  # (1,)

        # === Critic 更新 ===
        critic_loss = td_error.pow(2).mean()  # MSE

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=1.0)
        self.critic_optimizer.step()

        # === Actor 更新 ===
        # 关键：td_error.detach() — 不让 Critic 的梯度影响 Actor！
        actor_loss = -(action_log_prob * td_error.detach()).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=1.0)
        self.actor_optimizer.step()

        # 记录指标
        self.actor_losses.append(actor_loss.item())
        self.critic_losses.append(critic_loss.item())

        return td_error.item()

print("✅ ActorCritic 类定义完成")

## 5. 训练 on CartPole

In [ ]:
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

agent = ActorCritic(state_dim, n_actions, hidden_dim=128, actor_lr=3e-4, critic_lr=1e-3, gamma=0.99)

n_episodes = 400
episode_returns = []

for ep in range(n_episodes):
    state, _ = env.reset()
    done = False
    ep_return = 0

    while not done:
        action, log_prob = agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.update(state, log_prob, reward, next_state, terminated)
        state = next_state
        ep_return += reward

    episode_returns.append(ep_return)

    if (ep + 1) % 100 == 0:
        print(f"Episode {ep+1:4d} | Avg Return: {np.mean(episode_returns[-50:]):6.1f}")

print(f"\\n最终 50 episode 平均回报: {np.mean(episode_returns[-50:]):.1f}")

## 6. 训练曲线分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Returns
axes[0].plot(episode_returns, alpha=0.3, linewidth=0.5, color='steelblue')
window = 20
if len(episode_returns) > window:
    s = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(episode_returns)), s, linewidth=2, color='red')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')
axes[0].set_title('Actor-Critic — CartPole'); axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=500, color='green', linestyle='--', alpha=0.5)

# Actor Loss
axes[1].plot(agent.actor_losses, linewidth=0.5, alpha=0.7, color='coral')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Actor Loss')
axes[1].set_title('Actor Loss'); axes[1].grid(True, alpha=0.3)

# Critic Loss
axes[2].plot(agent.critic_losses, linewidth=0.5, alpha=0.7, color='purple')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Critic Loss')
axes[2].set_title('Critic Loss (MSE)'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/14_actor_critic.png', dpi=100); plt.close()
print("✅ 训练曲线已保存")

## 7. 关键理解：为什么 detach TD error？

如果不 detach：
```python
actor_loss = -(log_prob * td_error).mean()  # 危险！
```

这会导致 Actor 的梯度通过 td_error 回传到 Critic，而 td_error 中的 $V(s_{t+1})$ 又包含了网络参数。这意味着：
1. Actor 的更新会干扰 Critic 的更新
2. 梯度流混乱——两个网络互相影响
3. 实际效果通常更差

**正确做法**：
```python
actor_loss = -(log_prob * td_error.detach()).mean()  # 正确！
```

td_error.detach() 将 TD error 视为"常数信号"，Actor 只用它来缩放梯度方向。

## 8. Semi-Gradient 解释

**为什么不对 TD target 求梯度？**

TD target $r + \gamma V(s')$ 中包含 $V(s')$，它是 Critic 的输出。

如果对 target 中的 $V(s')$ 也求梯度：
- Critic 的 loss 变成了 $\|V(s) - (r + \gamma V(s'))\|^2$，其中 $V(s')$ 也是参数函数
- 但同时更新 $V(s)$ 和 $V(s')$ 会导致目标在移动
- Semi-gradient 只更新 $V(s)$，保持 $V(s')$ 固定（no_grad 或 detach）

这虽然引入了偏差，但保证了训练的稳定性。

## 9. 总结

| 方法 | Actor 更新信号 | 更新时机 | 方差 |
|------|----------------|----------|------|
| REINFORCE | $G_t$ | Episode 结束 | 高 |
| REINFORCE+Baseline | $G_t - V(s_t)$ | Episode 结束 | 中 |
| Actor-Critic | $\delta_t$ (TD error) | 每步 | 低 |

Actor-Critic 用 bootstrapping 实现在线学习，是后续所有高级算法的基础。

## 10. 练习
1. 去掉 td_error.detach()，训练是否能收敛？
2. 比较不同 critic_lr 对训练稳定性的影响
3. 实现 n-step Actor-Critic（n=5），与 1-step 比较
4. 添加 entropy bonus（-β·H(π)），观察对探索的影响

---
*下一节：[15_a2c.ipynb](15_a2c.ipynb) — A2C 从零实现*